# Aprendizado de Máquina — Aula prática 09

## Máquinas de Vetores de Suporte

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Os classificadores das Aulas 07 e 08 partiam de uma estimativa de
$P(Y = 1 \mid \mathbf{X} = \mathbf{x})$, e a classe prevista vinha de comparar essa
estimativa com um corte. As máquinas de vetores de suporte seguem outro caminho: procuram
diretamente uma fronteira que separe as classes, usando apenas a geometria das
observações de treino, sem modelar probabilidade alguma. Entre as retas que separam as
classes, escolhem a que fica mais longe das observações.

Este notebook acompanha as notas da aula, na mesma ordem e com a mesma notação. A Seção 2
constrói o classificador de margem máxima e verifica que ele depende apenas de algumas
observações, os vetores de suporte. A Seção 3 trata de classes que nenhuma reta separa,
com o classificador de vetor de suporte e o seu orçamento de violações. A Seção 4 obtém
fronteiras curvas ampliando as covariáveis e mostra como o truque do *kernel* faz isso sem
calcular as coordenadas novas. A Seção 5 escolhe por validação cruzada os dois parâmetros
da SVM com o *kernel* radial, e a Seção 6 mede o custo computacional do ajuste.

### Objetivos

Ao final deste notebook, você deve ser capaz de:

- reconhecer o classificador de margem máxima entre as retas que separam as classes,
  calcular a margem de uma reta e verificar que a solução depende apenas dos vetores de
  suporte;
- interpretar as folgas e o orçamento $C$ do classificador de vetor de suporte, prever o
  efeito do orçamento sobre a margem e o número de vetores de suporte e relacioná-lo ao
  argumento `C` do `SVC`;
- ver como a ampliação das covariáveis produz fronteiras não lineares e entender o truque
  do *kernel* e o papel de $\gamma$ no *kernel* radial;
- escolher `C` e $\gamma$ juntos, por validação cruzada numa grade bidimensional, e
  explicar por que a grade precisa ser bidimensional;
- medir como o tempo de ajuste do `SVC` cresce com o número de observações.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos do `scikit-learn` são as classes `SVC` e `LinearSVC`, de `sklearn.svm`,
que ajustam o classificador de vetor de suporte, a primeira com qualquer *kernel* e a
segunda só com fronteira linear, por outro algoritmo; o `FunctionTransformer`, que aplica
uma função a cada coordenada dentro de um `Pipeline`; e três geradores de dados
artificiais, `make_blobs`, `make_circles` e `make_moons`, cujas observações ficam no plano,
onde as fronteiras podem ser desenhadas.

In [ ]:
import sklearn.model_selection as skm
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC, LinearSVC

In [ ]:
import warnings
warnings.filterwarnings("ignore")

As duas funções a seguir desenham as figuras do notebook. A `desenhar_pontos` marca as
observações no plano, as da classe 0 em azul e as da classe 1 em vermelho, e desenha em
cor fraca as indicadas pela máscara `apagados`. A `desenhar_fronteira` calcula a função de
decisão do modelo numa grade fina de pontos do plano, pinta o fundo conforme o sinal dela e
desenha a fronteira, onde ela se anula. Com `margens=True`, desenha também, tracejadas, as
retas em que a função de decisão vale $-1$ e $+1$, que, para o `SVC`, são as bordas da
faixa da margem (Seção 3); com `suporte=True`, circula os vetores de suporte; e, com
`mapa=True`, pinta o fundo pelo valor da função de decisão, e não apenas pelo sinal.

In [ ]:
def desenhar_pontos(ax, X, y, titulo="", apagados=None):
    if apagados is None:
        apagados = np.zeros(len(y), dtype=bool)
    for classe, cor in [(0, "steelblue"), (1, "crimson")]:
        for selecao, alfa in [(apagados, 0.25), (~apagados, 1.0)]:
            pontos = X[(y == classe) & selecao]
            ax.scatter(pontos[:, 0], pontos[:, 1], s=18, color=cor, edgecolor="k", lw=0.3,
                       alpha=alfa)
    ax.set_xlim(X[:, 0].min() - 0.7, X[:, 0].max() + 0.7)
    ax.set_ylim(X[:, 1].min() - 0.7, X[:, 1].max() + 0.7)
    ax.set_title(titulo, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])


def desenhar_fronteira(ax, modelo, X, y, titulo="", margens=False, suporte=False,
                       apagados=None, mapa=False):
    passo = 0.02
    x0 = np.arange(X[:, 0].min() - 0.7, X[:, 0].max() + 0.7, passo)
    x1 = np.arange(X[:, 1].min() - 0.7, X[:, 1].max() + 0.7, passo)
    gx, gy = np.meshgrid(x0, x1)
    Z = modelo.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    if mapa:                    # a cor acompanha o valor da funcao de decisao, nao so o sinal
        lim = np.abs(Z).max()
        ax.pcolormesh(gx, gy, Z, shading="auto", cmap="RdBu_r", vmin=-lim, vmax=lim,
                      alpha=0.6)
    else:
        ax.contourf(gx, gy, Z > 0, levels=[-0.5, 0.5, 1.5],
                    colors=["steelblue", "crimson"], alpha=0.10)
    niveis = [-1, 0, 1] if margens else [0]
    estilos = ["--", "-", "--"] if margens else ["-"]
    ax.contour(gx, gy, Z, levels=niveis, colors="black", linestyles=estilos,
               linewidths=[1, 1.6, 1] if margens else [1.6])
    desenhar_pontos(ax, X, y, titulo, apagados)
    if suporte:
        sv = modelo.support_vectors_ if hasattr(modelo, "support_vectors_") \
            else modelo[-1].support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=110, facecolors="none", edgecolors="green",
                   linewidths=1.3)

---
## 2. O classificador de margem máxima e os vetores de suporte

Começamos pelo caso mais simples, o do classificador de margem máxima, em que uma reta
separa as duas classes. Os dados são 60 observações em $\mathbb{R}^2$, 30 de cada
classe, geradas pelo `make_blobs` em torno de dois centros que a própria função sorteia:
cada observação é o centro da sua classe somado a um vetor normal de coordenadas
independentes, com média zero e desvio-padrão 1,05. Antes de ajustar qualquer coisa,
olhamos para os dados. Na figura, o eixo horizontal é $x_1$ e o vertical é $x_2$; a
classe 0 está em azul, e a classe 1, em vermelho.

In [ ]:
X, y = make_blobs(n_samples=60, centers=2, cluster_std=1.05, random_state=6)

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_pontos(ax, X, y, "as 60 observacoes")

print(f"observacoes: {len(y)}  (classe 0: {np.sum(y == 0)}, classe 1: {np.sum(y == 1)})")

As duas nuvens não se misturam: há uma faixa vazia entre elas, e muitas retas deixam a
classe 0 inteira de um lado e a classe 1 inteira do outro. Para tornar isso preciso,
seguimos a notação das notas. Codificamos as classes por $y_i \in \{-1, +1\}$, com $-1$
para a classe 0 e $+1$ para a classe 1, e escrevemos cada reta como o conjunto dos
pontos em que se anula a função

$$f(\mathbf{x}) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 = \beta_0 + \boldsymbol{\beta}^\top \mathbf{x},
\qquad \boldsymbol{\beta} = (\beta_1, \beta_2)^\top \neq \mathbf{0}.$$

A reta é **separadora** quando cada observação está do lado em que $f$ tem o sinal da
sua classe, isto é, quando $y_i f(\mathbf{x}_i) > 0$ para $i = 1, \dots, n$. Como se
mostra nas notas, a distância de um ponto $\mathbf{x}$ à reta é
$|f(\mathbf{x})|/\|\boldsymbol{\beta}\|$, e, se a reta é separadora,
$|f(\mathbf{x}_i)| = y_i f(\mathbf{x}_i)$ para toda observação. A menor distância de uma
observação à reta é a **margem** da reta:

$$m(\beta_0, \boldsymbol{\beta}) = \min_{1 \le i \le n} \frac{y_i f(\mathbf{x}_i)}{\|\boldsymbol{\beta}\|}.$$

As notas definem a margem com $\|\boldsymbol{\beta}\| = 1$; dividir por
$\|\boldsymbol{\beta}\|$ dá o mesmo número sem normalizar os coeficientes.

A célula a seguir desenha três retas que escolhemos olhando a figura anterior, com
inclinações diferentes, e confere para cada uma se ela é separadora e qual é a sua
margem, calculada pela função `margem`. Cada reta é dada na forma $x_2 = a + b\,x_1$,
que é a reta $f(\mathbf{x}) = 0$ com $\beta_0 = a$, $\beta_1 = b$ e $\beta_2 = -1$. Com
esse sinal de $\beta_2$, $f$ é positiva abaixo da reta, que é onde está a classe $+1$.

In [ ]:
y_pm = 2 * y - 1        # classe 0 -> -1 e classe 1 -> +1


def margem(beta0, beta, X, y_pm):
    # menor valor de y_i f(x_i) / ||beta||, que e' a margem quando a reta separa as classes
    return np.min(y_pm * (beta0 + X @ beta)) / np.linalg.norm(beta)


retas = {"reta 1": (-1.5, -0.6), "reta 2": (-8.0, 0.25), "reta 3": (-6.9, 0.0)}   # (a, b)

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_pontos(ax, X, y, "tres retas separadoras")
extremos = np.array(ax.get_xlim())          # as duas pontas do eixo horizontal
for (nome, (a, b)), cor in zip(retas.items(), ["darkorange", "purple", "dimgray"]):
    ax.plot(extremos, a + b * extremos, color=cor, lw=1.6, label=nome)
ax.legend(fontsize=8, loc="upper right")

for nome, (a, b) in retas.items():
    beta0, beta = a, np.array([b, -1.0])
    separa = np.all(y_pm * (beta0 + X @ beta) > 0)
    print(f"{nome}: x2 = {a:.2f} {b:+.2f} x1   separa as classes? {separa}   "
          f"margem {margem(beta0, beta, X, y_pm):.4f}")

As três retas são separadoras: qualquer uma delas classifica corretamente as 60
observações de treino. As margens, porém, são diferentes: 0,5468 na reta 1, 0,7661 na
reta 2 e 0,4291 na reta 3, e cada uma é a distância da reta à observação mais próxima
dela. E essas três não são as únicas. As desigualdades $y_i f(\mathbf{x}_i) > 0$ são
estritas e em número finito, e por isso continuam valendo depois de qualquer mudança
suficientemente pequena de $(\beta_0, \boldsymbol{\beta})$: há infinitas retas
separadoras. É preciso um critério para escolher uma delas, e o das notas é ficar o mais
longe possível das observações de treino. O **classificador de margem máxima** é a reta
separadora de maior margem, isto é, a solução do problema

$$\max_{\beta_0,\, \boldsymbol{\beta},\, M} \ M \quad \text{sujeito a} \quad
\sum_{j=1}^{p} \beta_j^2 = 1 \quad \text{e} \quad
y_i\, (\beta_0 + \boldsymbol{\beta}^\top \mathbf{x}_i) \ge M, \ \ i = 1, \dots, n,$$

aqui com $p = 2$.

O `scikit-learn` não tem uma classe só para esse problema. O `SVC` com
`kernel="linear"` resolve um problema mais geral, o do classificador de vetor de
suporte, que é o assunto da Seção 3. Para estes dados, com `C=1000`, a solução que ele
devolve é a do classificador de margem máxima; o que o argumento `C` controla fica para
a Seção 3. A célula ajusta o `SVC` e calcula a margem da reta obtida com a mesma função
`margem`. Na figura, a reta cheia é o classificador, as tracejadas são as paralelas a
ele à distância $M$, o fundo mostra a classe que ele atribui a cada ponto do plano, e os
pontos circulados em verde são os **vetores de suporte**: as observações que estão à
distância $M$ do classificador.

In [ ]:
svm = SVC(kernel="linear", C=1000).fit(X, y)
beta0, beta = svm.intercept_[0], svm.coef_.ravel()

# distancia de cada observacao a reta (todas estao do lado certo dela)
distancia = y_pm * (beta0 + X @ beta) / np.linalg.norm(beta)
outras = np.setdiff1d(np.arange(len(y)), svm.support_)

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_fronteira(ax, svm, X, y, "classificador de margem maxima", margens=True, suporte=True)

print(f"reta: x2 = {-beta0 / beta[1]:.2f} {-beta[0] / beta[1]:+.2f} x1")
print(f"margem M = {margem(beta0, beta, X, y_pm):.4f}")
print(f"vetores de suporte: {svm.n_support_.sum()}, de indices {svm.support_}  (circulados em verde)")
print(f"   distancias a reta: {distancia[svm.support_].round(4)}")
print(f"menor distancia entre as outras {len(outras)} observacoes: {distancia[outras].min():.4f}")

A reta de margem máxima é $x_2 = -5{,}98 - 0{,}02\,x_1$, quase horizontal, e a sua
margem é $M = 1{,}1991$, maior que a de cada uma das três retas anteriores. Os vetores
de suporte são três: as observações 0 e 53, da classe $-1$, e a 49, da classe $+1$. As
distâncias delas à reta coincidem com $M$ até a terceira casa decimal; a diferença, de
alguns décimos de milésimo, vem de o `SVC` resolver o problema numericamente, com
tolerância de $10^{-3}$. A observação seguinte mais próxima já está a $1{,}3201$, e
nenhuma observação fica entre as tracejadas, como exigem as restrições do problema.

As notas afirmam que o classificador de margem máxima depende apenas dos vetores de
suporte: retirar uma observação que não é vetor de suporte não altera a solução. A
célula seguinte confere isso de duas maneiras. Primeiro, ajusta o classificador de novo
só com os três vetores de suporte, descartando as outras 57 observações; depois, com
todas as observações menos um dos vetores de suporte, a observação 0. Em cada caso,
imprime a reta obtida, a sua margem, os seus vetores de suporte e a maior diferença
entre os coeficientes novos e os originais, e desenha a nova solução. Na figura, a faixa
laranja é a reta original, as observações descartadas aparecem apagadas, e a observação
0, no segundo painel, está marcada com um x.

In [ ]:
coef_original = np.r_[svm.coef_.ravel(), svm.intercept_]
fora = svm.support_[0]                       # o vetor de suporte retirado no segundo caso
casos = {"so os 3 vetores de suporte": svm.support_,
         f"sem a observacao {fora}": np.delete(np.arange(len(y)), fora)}

fig, axes = subplots(1, 2, figsize=(10, 4.0))
for ax, (nome, usadas) in zip(axes, casos.items()):
    novo = SVC(kernel="linear", C=1000).fit(X[usadas], y[usadas])
    beta0, beta = novo.intercept_[0], novo.coef_.ravel()
    descartadas = ~np.isin(np.arange(len(y)), usadas)
    desenhar_fronteira(ax, novo, X, y, nome, margens=True, suporte=True, apagados=descartadas)
    # a reta original, larga e transparente, por baixo da nova
    extremos = np.array(ax.get_xlim())
    ax.plot(extremos, -(svm.intercept_[0] + svm.coef_[0, 0] * extremos) / svm.coef_[0, 1],
            color="darkorange", lw=6, alpha=0.4, zorder=1.5, label="reta original")
    ax.plot([], [], color="black", lw=1.6, label="nova solucao")
    ax.legend(fontsize=8, loc="lower right")

    print(f"{nome}:")
    print(f"   reta x2 = {-beta0 / beta[1]:.2f} {-beta[0] / beta[1]:+.2f} x1,  "
          f"M = {margem(beta0, beta, X[usadas], y_pm[usadas]):.4f}")
    print(f"   vetores de suporte: {usadas[novo.support_]}")
    print(f"   maior mudanca nos coeficientes: {np.abs(np.r_[beta, beta0] - coef_original).max():.2e}")
    if fora not in usadas:                   # o segundo caso: marca a observacao retirada
        ax.scatter(X[fora, 0], X[fora, 1], marker="x", s=70, color="black", lw=1.5, zorder=3)
        d = y_pm[fora] * (beta0 + X[fora] @ beta) / np.linalg.norm(beta)
        print(f"   a observacao {fora}, se voltasse, ficaria a {d:.4f} da reta, do lado certo")

Só com os três vetores de suporte, a solução é a mesma. Na figura, a reta nova passa sobre
a faixa laranja, e os números diferem dos originais só na segunda casa do intercepto da
reta ($-5{,}99$ contra $-5{,}98$) e na quarta casa da margem ($1{,}1988$ contra
$1{,}1991$); a maior diferença entre os coeficientes, $6{,}1 \cdot 10^{-3}$, é da ordem da
tolerância do otimizador. As 57 observações descartadas estavam todas a uma distância
maior que $M$ da reta, e as restrições delas não tinham efeito sobre a solução.

Sem a observação 0, a solução muda. A reta passa a ser $x_2 = -5{,}15 - 0{,}12\,x_1$, mais
inclinada que a original, a margem cresce para $1{,}2431$, e a observação 58 entra entre
os vetores de suporte, ao lado da 53 e da 49. A margem não poderia diminuir: sem uma
observação, o problema tem uma restrição a menos, e o máximo sobre um conjunto maior não é
menor. A observação 0, se voltasse, ficaria a $1{,}0691$ da reta nova, menos que a nova
margem, e a solução nova violaria a restrição dela; por isso, com a observação 0
presente, a solução tinha de ser outra. A maior diferença entre os coeficientes, $0{,}87$,
é mais de cem vezes a do primeiro caso.

É isso que as notas querem dizer quando afirmam que o classificador de margem máxima
depende apenas dos vetores de suporte. A propriedade tem dois lados. As observações longe
da fronteira não têm influência alguma, e a solução não muda se elas mudarem. Em
compensação, a solução fica nas mãos de pouquíssimas observações: uma observação nova
perto da fronteira, ou um erro de rotulagem numa das três, pode deslocá-la bastante, o que
as notas descrevem como variância alta. Na regressão logística, ao contrário, todas as
observações entram nas equações de máxima verossimilhança, ainda que as bem classificadas
e longe da fronteira entrem com peso pequeno.

---
## 3. O classificador de vetor de suporte e o orçamento $C$

O classificador de margem máxima só existe quando alguma reta separa as classes, e
dados reais raramente são assim. Nesta seção, as classes se misturam. Os dados são 40
observações, 20 de cada classe, geradas de duas normais bivariadas com coordenadas
independentes de desvio-padrão 0,85, uma centrada em $(-1;\,-0{,}5)$, para a classe 0, e
a outra em $(1;\,0{,}5)$, para a classe 1. Como na seção anterior, começamos olhando para
os dados.

In [ ]:
rng = np.random.default_rng(43)
Xm = np.vstack([rng.normal([-1.0, -0.5], 0.85, size=(20, 2)),
                rng.normal([1.0, 0.5], 0.85, size=(20, 2))])
ym = np.r_[np.zeros(20, dtype=int), np.ones(20, dtype=int)]

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_pontos(ax, Xm, ym, "as 40 observacoes")

print(f"observacoes: {len(ym)}  (classe 0: {np.sum(ym == 0)}, classe 1: {np.sum(ym == 1)})")

Desta vez as nuvens se tocam, e uma observação vermelha aparece no meio das azuis. Isso
sugere que nenhuma reta separa as classes, e o argumento a seguir prova isso sem depender
da impressão visual.

Cada lado de uma reta é um conjunto convexo: se dois pontos estão do mesmo lado, o
segmento entre eles também está. Por isso, se três pontos estão do mesmo lado, o
triângulo que eles formam fica inteiro desse lado. Uma reta que deixasse a classe 0
inteira de um lado deixaria desse lado, então, todo triângulo com vértices em
observações da classe 0, e também qualquer observação que estivesse dentro de um desses
triângulos. Basta, portanto, encontrar uma observação da classe 1 no interior de um
triângulo de observações da classe 0.

A célula a seguir faz essa busca. Um ponto $\mathbf{p}$ está no interior do triângulo de
vértices $\mathbf{a}$, $\mathbf{b}$ e $\mathbf{c}$ quando, percorrendo as arestas no
sentido $\mathbf{a} \to \mathbf{b} \to \mathbf{c} \to \mathbf{a}$, ele fica sempre do
mesmo lado de cada uma; o lado de $\mathbf{p}$ em relação à reta orientada de
$\mathbf{u}$ para $\mathbf{v}$ é dado pelo sinal de
$(v_1 - u_1)(p_2 - u_2) - (v_2 - u_2)(p_1 - u_1)$. A função `dentro` faz esse teste, e a
função `procurar_triangulo` percorre as observações da classe 1 e, para cada uma, os
triângulos de observações da classe 0, até achar um que a contenha.

In [ ]:
from itertools import combinations


def dentro(p, a, b, c):
    # p esta no interior do triangulo abc se fica do mesmo lado das arestas ab, bc e ca
    def lado(u, v):
        return np.sign((v[0] - u[0]) * (p[1] - u[1]) - (v[1] - u[1]) * (p[0] - u[0]))
    return lado(a, b) == lado(b, c) == lado(c, a) != 0


def procurar_triangulo(X, y):
    # uma observacao da classe 1 no interior de um triangulo de observacoes da classe 0
    for i in np.flatnonzero(y == 1):
        for t in combinations(np.flatnonzero(y == 0), 3):
            if dentro(X[i], *X[list(t)]):
                return int(i), [int(k) for k in t]


intrusa, triangulo = procurar_triangulo(Xm, ym)
print(f"a observacao {intrusa}, da classe 1, esta no interior do triangulo "
      f"das observacoes {triangulo}, da classe 0")

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_pontos(ax, Xm, ym, f"a observacao {intrusa} num triangulo da classe 0")
ax.fill(Xm[triangulo, 0], Xm[triangulo, 1], color="steelblue", alpha=0.2)
for k in [intrusa] + triangulo:
    ax.annotate(str(k), Xm[k], xytext=(4, 4), textcoords="offset points", fontsize=8)

A observação 39, da classe 1, está no interior do triângulo das observações 0, 1 e 2, da
classe 0, e por isso nenhuma reta separa as classes. O problema do classificador de
margem máxima não tem solução com $M > 0$, e o critério da Seção 2 não se aplica.

O classificador de vetor de suporte, das notas, contorna isso permitindo que algumas
observações violem a margem. A cada observação associa-se uma **folga**
$\varepsilon_i \ge 0$, e o problema passa a ser

$$\begin{aligned}
\max_{\beta_0,\, \boldsymbol{\beta},\, \boldsymbol{\varepsilon},\, M}\ \ & M \\
\text{sujeito a}\ \ & \textstyle\sum_{j=1}^{p} \beta_j^2 = 1, \\
& y_i\,(\beta_0 + \boldsymbol{\beta}^\top \mathbf{x}_i) \ge M(1 - \varepsilon_i), \quad i = 1, \dots, n, \\
& \varepsilon_i \ge 0, \quad i = 1, \dots, n, \qquad \textstyle\sum_{i=1}^{n} \varepsilon_i \le C,
\end{aligned}$$

em que $C \ge 0$ é o **orçamento** de violações. Numa solução, cada folga vale
$\varepsilon_i = \max\{0,\ 1 - y_i f(\mathbf{x}_i)/M\}$, e o seu valor diz onde está a
observação: $\varepsilon_i = 0$ se ela está do lado certo da margem ou sobre ela;
$0 < \varepsilon_i < 1$ se está dentro da faixa, mas do lado certo do hiperplano;
$\varepsilon_i = 1$ se está sobre o hiperplano; e $\varepsilon_i > 1$ se está do lado
errado do hiperplano, isto é, se é classificada errado. Com $C = 0$, todas as folgas são
nulas, e o problema é o do classificador de margem máxima. Quanto maior o orçamento, mais
violações são toleradas, e a margem $M$ ótima não diminui. Os **vetores de suporte** são
as observações sobre a margem ou do lado errado dela, isto é, com
$y_i f(\mathbf{x}_i) \le M$.

O `SVC` resolve esse problema, mas não recebe o orçamento. O argumento `C` dele é outro
parâmetro, que age no sentido oposto: quanto maior o argumento, menor, ou igual, o
orçamento. Pode-se mostrar que, para cada valor do argumento, a solução devolvida pelo
`SVC` é a solução do problema acima com orçamento igual à soma das folgas dessa solução.
Para saber com que orçamento estamos trabalhando, então, ajustamos o `SVC` e somamos as
folgas.

Para calcular as folgas, é preciso saber em que escala o `SVC` devolve os coeficientes.
Ele devolve $\mathbf{w}$ (`coef_`) e $b$ (`intercept_`) numa escala em que as tracejadas
são as retas em que $b + \mathbf{w}^\top\mathbf{x}$ vale $-1$ e $+1$. Na normalização das
notas, $\boldsymbol{\beta} = \mathbf{w}/\|\mathbf{w}\|$ e $\beta_0 = b/\|\mathbf{w}\|$; a
distância do hiperplano a cada tracejada é $M = 1/\|\mathbf{w}\|$; e
$y_i f(\mathbf{x}_i)/M = y_i\,(b + \mathbf{w}^\top\mathbf{x}_i)$. A função
`margem_e_folgas` faz essas contas.

A célula a seguir ajusta o `SVC` com o argumento `C=0.5`, imprime as observações de folga
positiva e escreve, ao lado de cada uma, a sua folga. Folgas menores que 0,001 não
contam: são de observações sobre a margem, com folga nula, que o otimizador, de
tolerância $10^{-3}$, deixa um pouco para dentro da faixa.

In [ ]:
ym_pm = 2 * ym - 1        # classe 0 -> -1 e classe 1 -> +1


def margem_e_folgas(modelo, X, y_pm):
    # M, os numeros y_i f(x_i) / M e as folgas das notas, a partir de coef_ e intercept_
    w, b = modelo.coef_.ravel(), modelo.intercept_[0]
    razao = y_pm * (b + X @ w)
    return 1 / np.linalg.norm(w), razao, np.maximum(0, 1 - razao)


modelo = SVC(kernel="linear", C=0.5).fit(Xm, ym)
M, razao, folga = margem_e_folgas(modelo, Xm, ym_pm)
violam = np.flatnonzero(folga > 1e-3)

fig, ax = subplots(figsize=(5.0, 4.0))
desenhar_fronteira(ax, modelo, Xm, ym, f"as folgas (soma {folga.sum():.2f})", margens=True)
for k in violam:
    ax.annotate(f"{folga[k]:.2f}", Xm[k], xytext=(4, 4), textcoords="offset points", fontsize=7)

print(f"M = {M:.4f}")
print(f"orcamento = soma das folgas = {folga.sum():.4f}")
print(f"do lado errado do hiperplano (folga > 1): {np.sum(folga > 1)}")
print("vetores de suporte do SVC = observacoes com y f(x)/M <= 1 (a menos de 0,001)? "
      f"{np.array_equal(np.sort(modelo.support_), np.flatnonzero(razao <= 1 + 1e-3))}")
pd.DataFrame({"classe": ym_pm[violam], "y f(x) / M": razao[violam], "folga": folga[violam]},
             index=pd.Index(violam, name="observacao")).sort_values("folga", ascending=False).round(4)

Com o argumento `C=0.5`, a margem é $M = 0{,}6292$ e o orçamento usado é $4{,}8340$, a
soma das oito folgas da tabela. A observação 39 tem folga $2{,}4647 > 1$: está do lado
errado do hiperplano e é a única observação de treino classificada errado. Como garante
a proposição das notas, o número de erros de treino, 1, é menor que o orçamento. As
outras sete têm folgas entre 0 e 1: estão dentro da faixa, mas do lado certo do
hiperplano. As 32 observações restantes estão
do lado certo da margem e não gastam orçamento algum. Os vetores de suporte do `SVC` são
exatamente as observações com $y_i f(\mathbf{x}_i)/M \le 1$, como na definição das notas:
as oito da tabela e três que estão sobre as tracejadas.

Para ver o papel do orçamento, ajustamos o `SVC` com vários valores do argumento `C` e,
para cada um, calculamos o orçamento, a margem $M$, o número de violações (observações
com folga positiva), o número de erros de treino (observações do lado errado do
hiperplano) e o número de vetores de suporte. A figura mostra três desses ajustes, com os
argumentos 100, 0,5 e 0,05, que correspondem a orçamentos crescentes. Para não carregar a
figura, os vetores de suporte aparecem em cor cheia, e as demais observações, apagadas.

In [ ]:
linhas = []
for c in [1000, 100, 1, 0.5, 0.1, 0.05]:
    modelo = SVC(kernel="linear", C=c).fit(Xm, ym)
    M, _, folga = margem_e_folgas(modelo, Xm, ym_pm)
    linhas.append({"argumento C": c, "orcamento": folga.sum(), "M": M,
                   "violacoes": int(np.sum(folga > 1e-3)),
                   "erros de treino": int(np.sum(folga > 1)),
                   "vetores de suporte": len(modelo.support_)})

fig, axes = subplots(1, 3, figsize=(11, 3.6))
for ax, c in zip(axes, [100, 0.5, 0.05]):
    modelo = SVC(kernel="linear", C=c).fit(Xm, ym)
    M, _, folga = margem_e_folgas(modelo, Xm, ym_pm)
    desenhar_fronteira(ax, modelo, Xm, ym, f"orcamento {folga.sum():.2f},  M = {M:.2f}",
                       margens=True,
                       apagados=~np.isin(np.arange(len(ym)), modelo.support_))

pd.DataFrame(linhas).set_index("argumento C").round(4)

Da esquerda para a direita, o orçamento cresce de 3,11 para 4,83 e para 10,82, e a faixa
se alarga: $M$ passa de 0,34 para 0,63 e para 1,42. A faixa mais larga alcança mais
observações: as que violam a margem passam de 2 para 8 e para 19, e os vetores de
suporte, de 5 para 11 e para 22. A tabela mostra a mesma tendência com mais valores do
argumento: quanto menor o argumento, maior o orçamento, e a margem nunca diminui quando o
orçamento aumenta, como as notas demonstram. Em todos os ajustes há uma única observação
do lado errado do hiperplano, e, como garante a proposição das notas, esse número é
menor que o orçamento.

Aumentar o argumento de 100 para 1000 praticamente não muda a solução: o orçamento fica
em 3,11, e $M$, em 0,34. O orçamento não poderia chegar a zero: como as classes não são
separáveis, toda reta deixa ao menos uma observação do lado errado, e a folga dessa
observação passa de 1. Por isso o classificador de margem máxima, o caso $C = 0$, não
existe aqui.

O orçamento regula o balanço entre viés e variância. Com orçamento pequeno, a faixa é
estreita, e a solução depende de poucas observações, as mais próximas da fronteira: 5, no
primeiro painel. Mudar uma delas pode mudar bastante a solução: o viés é baixo e a
variância, alta. Com orçamento grande, a solução depende de muitas observações, 22 no
último painel, e muda pouco quando uma delas muda: o viés é mais alto e a variância, mais
baixa. Como os outros parâmetros de ajuste do curso, o orçamento é escolhido por
validação cruzada (Aula 03), e a Seção 5 faz isso por meio do argumento `C`.

---
## 4. A ampliação das covariáveis e o truque do *kernel*

O classificador de vetor de suporte tem fronteira linear, e nenhuma reta separa bem duas
classes quando uma cerca a outra. Os dados desta seção são os anéis concêntricos da
Figura 2 das notas: 350 observações, 175 de cada classe, geradas pelo `make_circles`. A
classe 1 fica no anel interno, de raio 0,42, e a classe 0 no externo, de raio 1, as duas
com ruído normal de desvio-padrão 0,13 em cada coordenada.

As notas sugerem ampliar o espaço das covariáveis, como na regressão polinomial. Para os
anéis, os quadrados bastam: a circunferência $x_1^2 + x_2^2 = r^2$, que separaria as
classes, é uma reta no plano de coordenadas $z_1 = x_1^2$ e $z_2 = x_2^2$, a reta
$z_1 + z_2 = r^2$. As notas acrescentam os quadrados às covariáveis originais, ficando com
$2p$ covariáveis; aqui usamos só os quadrados, e a vantagem é que o espaço ampliado
continua sendo um plano, que se pode desenhar.

A célula a seguir ajusta duas SVMs lineares: uma às covariáveis originais e outra aos
quadrados, por meio de um `Pipeline` em que o `FunctionTransformer(np.square)` eleva cada
coordenada ao quadrado antes do `SVC`. A figura mostra, da esquerda para a direita, a reta
ajustada no plano original, a reta ajustada no plano dos quadrados e essa mesma reta
desenhada de volta no plano original.

In [ ]:
Xc, yc = make_circles(n_samples=350, factor=0.42, noise=0.13, random_state=3)

linear = SVC(kernel="linear").fit(Xc, yc)
quadrados = Pipeline([("quadrados", FunctionTransformer(np.square)),
                      ("svm", SVC(kernel="linear"))]).fit(Xc, yc)

fig, axes = subplots(1, 3, figsize=(11, 3.6))
desenhar_fronteira(axes[0], linear, Xc, yc,
                   f"plano original, reta\ntreino {linear.score(Xc, yc):.3f}")
desenhar_fronteira(axes[1], quadrados[-1], Xc ** 2, yc,
                   f"plano (x1^2, x2^2), reta\ntreino {quadrados.score(Xc, yc):.3f}")
desenhar_fronteira(axes[2], quadrados, Xc, yc,
                   f"a mesma reta, no plano original\ntreino {quadrados.score(Xc, yc):.3f}")

w, b = quadrados[-1].coef_.ravel(), quadrados[-1].intercept_[0]
print(f"no plano dos quadrados, a reta {b:.4f} {w[0]:+.4f} z1 {w[1]:+.4f} z2 = 0")
print(f"no plano original, a elipse x1^2 / {-b / w[0]:.4f} + x2^2 / {-b / w[1]:.4f} = 1, "
      f"de semieixos {np.sqrt(-b / w[0]):.4f} e {np.sqrt(-b / w[1]):.4f}")

No plano original, a melhor reta acerta 60% das observações de treino, pouco mais que os
50% de quem atribui todas à mesma classe. No plano dos quadrados, os anéis ficam de lados
opostos de uma reta, e a SVM linear acerta 99,4%. Trazida de volta ao plano original, a
reta $b + w_1 z_1 + w_2 z_2 = 0$ vira a curva $b + w_1 x_1^2 + w_2 x_2^2 = 0$, que é a
elipse impressa pela célula: semieixos 0,7387 e 0,7361, quase uma circunferência de raio
0,74, entre os raios dos dois anéis. É o que as notas afirmam: a fronteira é um hiperplano
no espaço ampliado e uma cônica no original.

O preço da ampliação é o número de coordenadas. Com todos os monômios de grau até $q$ nas
$p$ covariáveis, são $\binom{p+q}{q}$ coordenadas: 6 para $p = q = 2$, mas já 231 para
$p = 20$ e $q = 2$, e $53\,130$ para $p = 20$ e $q = 5$. O truque do *kernel* contorna esse
custo. Pela proposição das notas, a solução depende dos dados apenas por meio dos produtos
internos $\langle \mathbf{x}_i, \mathbf{x}_j \rangle$; depois da ampliação, por meio dos
produtos internos $\langle \varphi(\mathbf{x}_i), \varphi(\mathbf{x}_j) \rangle$, em que
$\varphi$ leva cada ponto às suas coordenadas ampliadas. Um *kernel* calcula esses números
sem construir $\varphi$. Para o *kernel* polinomial de grau 2 em $\mathbb{R}^2$, expandir o
quadrado dá

$$k(\mathbf{x}, \mathbf{x}') = (1 + x_1x_1' + x_2x_2')^2 = \langle \varphi(\mathbf{x}), \varphi(\mathbf{x}') \rangle,
\qquad \varphi(\mathbf{x}) = \big(1,\ \sqrt{2}\,x_1,\ \sqrt{2}\,x_2,\ x_1^2,\ \sqrt{2}\,x_1x_2,\ x_2^2\big),$$

e $\varphi$ tem uma coordenada para cada um dos 6 monômios de grau até 2. A SVM com esse
*kernel* é, portanto, a SVM linear ajustada aos pontos $\varphi(\mathbf{x}_i)$, e, para
ajustá-la, bastam os $n(n+1)/2$ valores $k(\mathbf{x}_i, \mathbf{x}_j)$, cada um calculado
com um produto interno em $\mathbb{R}^2$, uma soma e um quadrado. Em geral, com $p$
covariáveis e grau $q$, $\varphi$ teria $\binom{p+q}{q}$ coordenadas, enquanto
$k(\mathbf{x}, \mathbf{x}') = (1 + \langle \mathbf{x}, \mathbf{x}' \rangle)^q$ continua
custando cerca de $p$ operações.

O *kernel* mais usado é o radial, ou gaussiano,

$$k(\mathbf{x}, \mathbf{x}') = \exp\big(-\gamma\,\|\mathbf{x} - \mathbf{x}'\|^2\big), \qquad \gamma > 0.$$

Ele depende dos dois pontos apenas pela distância $d = \|\mathbf{x} - \mathbf{x}'\|$ entre
eles: vale 1 quando os pontos coincidem e decresce para 0 à medida que eles se afastam. É,
portanto, uma medida de semelhança, e $\gamma$ fixa a escala: $k = 1/2$ quando
$\gamma d^2 = \ln 2$, isto é, à distância $d = \sqrt{\ln 2/\gamma}$. Com $\gamma = 0{,}5$,
essa distância é 1,18, maior que a distância de 0,58 entre os raios dos anéis, e uma
observação ainda é parecida com as do outro anel; com $\gamma = 50$, ela é 0,12, a escala
do ruído, e cada observação só é parecida com as suas vizinhas imediatas. O $\varphi$ do
*kernel* radial que as notas exibem tem infinitas coordenadas: para $p = 1$, pela série de
Taylor de $e^{2\gamma x x'}$,

$$e^{-\gamma (x - x')^2} = \sum_{m=0}^{\infty} \varphi_m(x)\,\varphi_m(x'), \qquad
\varphi_m(x) = \sqrt{\frac{(2\gamma)^m}{m!}}\; x^m e^{-\gamma x^2}.$$

Esse $\varphi$ não pode ser construído explicitamente, e, ainda assim, $k(x, x')$ se
calcula com uma subtração, um quadrado e uma exponencial. No `SVC`, esse *kernel* é o
`kernel="rbf"`, e o argumento `gamma` é o $\gamma$ da fórmula.

Com o *kernel* radial, a função de decisão das notas,

$$f(\mathbf{x}) = \beta_0 + \sum_{i \in \mathcal{S}} \alpha_i\, k(\mathbf{x}, \mathbf{x}_i),$$

é uma soma de funções em forma de sino centradas nos vetores de suporte, com sinal e
altura dados pelos $\alpha_i$ e largura governada por $\gamma$. A célula a seguir ajusta a
SVM com $\gamma = 0{,}5$ e com $\gamma = 50$ e pinta cada ponto do plano conforme o valor de
$f$: azul onde $f$ é negativa, vermelho onde é positiva, tanto mais escuro quanto maior
$|f|$. A linha preta é a fronteira $f = 0$, e os vetores de suporte aparecem em cor cheia.
A célula imprime também $\beta_0$ e o valor de $f$ no ponto $(2, 2)$, longe de todas as
observações.

In [ ]:
fig, axes = subplots(1, 2, figsize=(10, 4.2))
for ax, g in zip(axes, [0.5, 50]):
    modelo = SVC(kernel="rbf", gamma=g).fit(Xc, yc)
    desenhar_fronteira(ax, modelo, Xc, yc,
                       f"RBF, gamma = {g}\ntreino {modelo.score(Xc, yc):.3f}, "
                       f"{len(modelo.support_)} vetores de suporte",
                       mapa=True, apagados=~np.isin(np.arange(len(yc)), modelo.support_))
    print(f"gamma = {g:4}: beta0 = {modelo.intercept_[0]:+.4f},  "
          f"f(2, 2) = {modelo.decision_function([[2.0, 2.0]])[0]:+.4f}")

Com $\gamma = 0{,}5$, $f$ varia devagar: é positiva no centro, decresce para fora, e a
fronteira é uma curva suave, próxima de uma circunferência. Com $\gamma = 50$, cada termo
$\alpha_i k(\mathbf{x}, \mathbf{x}_i)$ só pesa a distâncias da ordem de 0,1 do seu vetor de
suporte. Longe de todos eles, $f$ é praticamente constante e igual a $\beta_0$: no ponto
$(2, 2)$, $f$ coincide com $\beta_0 = -0{,}3972$ em todas as casas impressas, e é essa a cor
uniforme do fundo. Perto das observações, $f$ é feita de sinos estreitos, e a fronteira
contorna observações isoladas; para isso, a solução usa 194 das 350 observações como
vetores de suporte, contra 84 com $\gamma = 0{,}5$.

As duas acertam quase o mesmo no treino, 99,4% e 99,1%, e a acurácia de treino não revela
que a segunda fronteira é muito mais irregular e se ajusta ao ruído. O $\gamma$ do
*kernel* radial controla a flexibilidade da fronteira, papel que o grau do polinômio teve
na Aula 01 e o número de vizinhos do KNN, na Aula 04: com $\gamma$ grande, a fronteira
pode acompanhar cada observação, com risco de superajuste. Para escolher $\gamma$ é
preciso estimar o risco, por validação cruzada, por exemplo, e a próxima seção faz isso
junto com o argumento `C`.

---
## 5. A escolha conjunta de `C` e $\gamma$ por validação cruzada

A SVM com o *kernel* radial tem dois parâmetros de ajuste: o argumento `C` do `SVC`, que
determina o orçamento de violações da Seção 3, em sentido oposto a ele, e o $\gamma$ do
*kernel*, que determina a flexibilidade da fronteira (Seção 4). Os dois agem sobre a
complexidade da fronteira e por isso interagem: o efeito de um $\gamma$ maior, que torna a
fronteira mais flexível, pode ser compensado por um `C` menor, que aumenta o orçamento e a
suaviza. Então o melhor `C` para um valor de $\gamma$ não precisa ser o melhor para outro,
e a escolha tem de ser feita sobre os pares $(C, \gamma)$.

Os dados são 400 observações do `make_moons`: duas meias-luas entrelaçadas, uma por classe,
com ruído normal de desvio-padrão 0,3 em cada coordenada. A célula a seguir estima, por
validação cruzada com 5 dobras (Aula 03), a acurácia da SVM com o *kernel* radial em cada
um dos $12 \times 12 = 144$ pares de uma grade, com `C` de $10^{-2}$ a $10^{3}$ e $\gamma$
de $10^{-2}$ a $10^{2}$, igualmente espaçados em escala logarítmica. As covariáveis são
padronizadas dentro do `Pipeline` (Aula 06): o *kernel* radial depende de
$\|\mathbf{x} - \mathbf{x}'\|^2 = \sum_j (x_j - x_j')^2$, e, sem padronização, a covariável
de maior dispersão dominaria essa soma, e o valor adequado de $\gamma$ dependeria das
unidades de medida. A célula desenha o mapa das acurácias, com o melhor par marcado por uma
estrela, e imprime, para cada $\gamma$ da grade, o melhor `C` e a acurácia correspondente.

In [ ]:
Xg, yg = make_moons(n_samples=400, noise=0.3, random_state=5)
Cs = np.logspace(-2, 3, 12)
gammas = np.logspace(-2, 2, 12)

busca = skm.GridSearchCV(Pipeline([("sc", StandardScaler()), ("svc", SVC())]),
                         {"svc__C": Cs, "svc__gamma": gammas},
                         cv=5, scoring="accuracy", n_jobs=-1).fit(Xg, yg)
grade = busca.cv_results_["mean_test_score"].reshape(len(Cs), len(gammas))   # linhas: C

fig, ax = subplots(figsize=(5.2, 4.0))
im = ax.pcolormesh(gammas, Cs, grade, shading="auto", cmap="viridis")
ax.plot(busca.best_params_["svc__gamma"], busca.best_params_["svc__C"], "r*", ms=16)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("gamma"); ax.set_ylabel("C")
ax.set_title("acuracia de validacao cruzada (a estrela e' o melhor par)", fontsize=9)
fig.colorbar(im, ax=ax)

print(f"melhor par: C = {busca.best_params_['svc__C']:.4g}, "
      f"gamma = {busca.best_params_['svc__gamma']:.4g}, acuracia {busca.best_score_:.4f}\n")
pd.DataFrame({"gamma": gammas, "melhor C": Cs[grade.argmax(axis=0)],
              "acuracia": grade.max(axis=0)}).round(4)

O melhor par da grade é $C = 43{,}3$ e $\gamma = 0{,}2848$, com acurácia de validação
cruzada de 0,8975. No mapa, as acurácias próximas da máxima não se concentram num ponto
nem se distribuem numa cruz: formam uma faixa diagonal, que desce da esquerda para a
direita. A tabela mostra o mesmo em números. Nas colunas de $\gamma$ entre 0,12 e 3,5, onde
estão as melhores acurácias, o melhor `C` diminui à medida que $\gamma$ cresce: 351 com
$\gamma = 0{,}12$, 43,3 com $\gamma = 0{,}28$, 5,3 com $\gamma = 0{,}66$, 0,23 com
$\gamma = 1{,}52$ e 0,081 com $\gamma = 3{,}51$. É a compensação descrita acima: uma
fronteira mais flexível pede um orçamento maior. Para $\gamma$ maior que 3,5, a melhor
acurácia de cada coluna diminui, até 0,8175 com $\gamma = 100$: a fronteira se ajusta ao
ruído, como na Seção 4.

Por isso não basta fixar um dos parâmetros e buscar só o outro: o resultado depende do valor
fixado. Com $\gamma = 0{,}01$, nenhum `C` da grade passa de 0,8425, e, com $\gamma = 100$,
de 0,8175. O $\gamma$ padrão do `SVC`, `gamma="scale"`, vale $1/(p\,v)$, em que $v$ é a
variância de todas as entradas da matriz de covariáveis; depois da padronização, $v = 1$
e, com $p = 2$, ele vale $1/2$. Neste exemplo, esse valor cai sobre a faixa, entre as
colunas de $\gamma = 0{,}28$ e $\gamma = 0{,}66$, cujas melhores acurácias são 0,8975 e
0,8925. Em outros dados, nada garante isso, e a grade bidimensional não depende dessa
coincidência.

---
## 6. O custo computacional do `SVC`

O `SVC` encontra $\beta_0$ e os $\alpha_i$ da Seção 4 resolvendo um problema de otimização
quadrática com uma variável para cada observação de treino, a partir dos $n(n+1)/2$
valores $k(\mathbf{x}_i, \mathbf{x}_j)$. Segundo a documentação do `scikit-learn`, o tempo
desse ajuste cresce entre $n^2$ e $n^3$, conforme os dados. O `LinearSVC`, que só ajusta
fronteiras lineares, usa outro algoritmo, que a documentação descreve como quase linear em
$n$; a regressão logística entra como referência.

A célula a seguir mede o tempo de ajuste dos três métodos em amostras do `make_moons` de
tamanhos $n$ = 2 000, 4 000, 8 000, 16 000 e 32 000. Se o tempo se comporta como
$t \approx a\,n^b$, então $\log t \approx \log a + b \log n$, e o expoente $b$ é a
inclinação da reta de mínimos quadrados de $\log t$ contra $\log n$; a célula estima $b$
assim e desenha os tempos em escala logarítmica nos dois eixos. Os tempos dependem da
máquina e variam de uma execução para outra, e só a ordem de grandeza deles importa.

In [ ]:
import time

tamanhos = np.array([2000, 4000, 8000, 16_000, 32_000])
tempos = {"SVC (kernel RBF)": [], "LinearSVC": [], "logistica": []}
for n in tamanhos:
    Xs, ys = make_moons(n_samples=int(n), noise=0.35, random_state=1)
    Xs = StandardScaler().fit_transform(Xs)
    for nome, m in [("SVC (kernel RBF)", SVC(C=1, gamma="scale")),
                    ("LinearSVC", LinearSVC(C=1, max_iter=20000)),
                    ("logistica", LogisticRegression(max_iter=5000))]:
        t0 = time.perf_counter(); m.fit(Xs, ys); tempos[nome].append(time.perf_counter() - t0)

fig, ax = subplots(figsize=(5.2, 3.2))
for nome, ts in tempos.items():
    ax.plot(tamanhos, ts, "o-", ms=4, label=nome)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("n"); ax.set_ylabel("segundos para ajustar")
ax.legend(fontsize=8)

for nome, ts in tempos.items():
    expoente = np.polyfit(np.log(tamanhos), np.log(np.maximum(ts, 1e-6)), 1)[0]
    print(f"{nome:18s} expoente estimado: {expoente:.2f}   tempo: {ts[0] * 1000:7.1f} ms "
          f"em n = 2000 e {ts[-1] * 1000:7.1f} ms em n = 32000")

Para o `SVC`, o expoente estimado fica perto de 2, de acordo com a documentação, e, em
$n = 32\,000$, um ajuste leva alguns segundos. O `LinearSVC` e a regressão logística levam
alguns milissegundos, centenas de vezes menos. Para esses dois, o expoente estimado, bem
abaixo de 1, não deve ser lido como a taxa de crescimento do tempo: entre $n = 2\,000$ e
$n = 32\,000$, com dezesseis vezes mais observações, o tempo deles cresce bem menos que
dezesseis vezes, sinal de que, nesses tamanhos, uma parte do tempo de ajuste não depende de
$n$ e ainda pesa no total.

O expoente do `SVC` permite estimar o que acontece com mais dados. Se o tempo cresce como
$n^2$, passar de $n = 32\,000$ para $n = 10^6$ multiplica o tempo por
$(10^6/32\,000)^2 \approx 980$: um ajuste de alguns segundos passa a levar cerca de uma
hora. E isso vale para um único ajuste; a busca da Seção 5 faz um ajuste para cada par da
grade e cada dobra da validação cruzada, $144 \times 5 = 720$ ajustes.

Para $n$ grande, há três saídas: o `LinearSVC`, quando uma fronteira linear basta; uma
aproximação do *kernel*, como o `Nystroem` do `scikit-learn`, que constrói explicitamente
da ordem de centenas de coordenadas cujo produto interno aproxima o *kernel* radial, de
modo que um classificador linear nessas coordenadas faz as vezes da SVM; ou um método de
outra família.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| vetores de suporte | §2 | refeito só com os 3 vetores de suporte, o classificador não muda; sem um deles, a mudança nos coeficientes é mais de cem vezes maior |
| o orçamento $C$ | §3 | orçamento maior, margem maior ou igual; nestes dados, também mais violações e mais vetores de suporte; o argumento `C` do `SVC` age no sentido oposto |
| ampliação | §4 | uma reta no plano $(x_1^2, x_2^2)$ é uma elipse no plano original; a acurácia de treino passa de 60% a 99,4% |
| truque do *kernel* | §4 | o *kernel* calcula os produtos internos das coordenadas ampliadas sem construí-las; o $\varphi$ do radial tem infinitas coordenadas |
| $\gamma$ do radial | §4 | $f$ é uma soma de sinos centrados nos vetores de suporte; com $\gamma$ grande, a fronteira contorna observações isoladas, e longe delas $f \approx \beta_0$ |
| grade bidimensional | §5 | as melhores acurácias formam uma faixa diagonal: $\gamma$ maior pede `C` menor, e por isso os dois são escolhidos juntos |
| custo | §6 | o tempo de ajuste do `SVC` cresce com expoente perto de 2; em $n = 32\,000$, são segundos, contra milissegundos do `LinearSVC` |

**Leitura recomendada.** [AME], §8.2 (SVM) e §4.6 (o espaço de Hilbert com *kernel*
reprodutor, que formaliza o truque do *kernel*). [ISLP], Capítulo 9: §9.1, o classificador
de margem máxima; §9.2, o classificador de vetor de suporte, em que o $C$ é o orçamento das
notas e age, portanto, no sentido oposto ao argumento `C` do `scikit-learn`; e §9.3, os
*kernels*.

**Para praticar.** `Lista teorica 09.pdf` (teórica, com gabarito) e
`Lista prática 09.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 10 completa a parte de classificação com o KNN e as árvores de
classificação e compara, nos mesmos dados, as fronteiras dos classificadores vistos no
curso, das mais rígidas às mais flexíveis.